In [1]:

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=False
# )

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

# tokenizer = AutoTokenizer.from_pretrained("EleutherAI/llemma_7b")
# model = AutoModelForCausalLM.from_pretrained("EleutherAI/llemma_7b", quantization_config=quant_config, device_map={"": 0})


# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quant_config,
#                                              device_map={"": 0})


tokenizer = AutoTokenizer.from_pretrained('deepseek-ai/deepseek-math-7b-base')
model = AutoModelForCausalLM.from_pretrained('deepseek-ai/deepseek-math-7b-base', quantization_config=quant_config, device_map='auto')


# filename = 'deepseek-math-7b-rl.Q8_0.gguf'
# tokenizer = AutoTokenizer.from_pretrained('QuantFactory/deepseek-math-7b-rl-GGUF')#, gguf_file=filename)
# model = AutoModelForCausalLM.from_pretrained('QuantFactory/deepseek-math-7b-rl-GGUF', gguf_file=filename, device_map={"": 0})




Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Could not load bitsandbytes native library: libcusparse.so.11: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/home/sean/Documents/venvs/bait/lib/python3.10/site-packages/bitsandbytes/cextension.py", line 109, in <module>
    lib = get_native_library()
  File "/home/sean/Documents/venvs/bait/lib/python3.10/site-packages/bitsandbytes/cextension.py", line 96, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
  File "/usr/lib/python3.10/ctypes/__init__.py", line 452, in LoadLibrary
    return self._dlltype(name)
  File "/usr/lib/python3.10/ctypes/__init__.py", line 374, in __init__
    self._handle = _dlopen(self._name, mode)
OSError: libcusparse.so.11: cannot open shared object file: No such file or directory

CUDA Setup failed despite CUDA being available. Please run the following command to get mo

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/sean/Documents/venvs/bait/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


AttributeError: 'NoneType' object has no attribute 'cget_col_row_stats'

In [ ]:
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# state = '\"\"\"Given the Lean 4 tactic state, suggest a next tactic. Do NOT show working"\n\n\
state = 'α : Type u_1\n\
r : α → α → Prop\n\
inst1 : DecidableEq α\n\
inst : IsIrrefl α r\n\
⊢ CutExpand r ≤ InvImage (Finsupp.Lex (cr Π fun x x_1 => x̸ = x_1)\n\
fun x x_1 => x < x_1) ↑toFinsupp\n\
---\n\n\
Next tactic:'

tokenized_state = tokenizer(
    state,
    padding="longest",
    max_length=2300,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()
state_mask = tokenized_state.attention_mask.cuda()


In [ ]:
with torch.no_grad():
    num_samples = 10
    output = model.generate(
        input_ids=state_ids,
        attention_mask=state_mask,
        max_new_tokens=50,
        top_k =50,
        top_p = 0.95,
        num_return_sequences=num_samples,
        do_sample=True,
        # length_penalty=1.0
    )

In [ ]:
tokenizer.batch_decode(output)